In [169]:
import pandas as pd

df1 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_1.csv')
df2 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_2.csv')
df3 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_3.csv')

df = pd.concat([df1, df2, df3], axis = 0)

df = df.drop(columns = ['jerseyNum', 'comment'], axis = 1)
df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())
df = df.dropna()
df['minutes'] = df['minutes'].astype(str)
df['minutes'] = df['minutes'].apply(lambda x: int(x.split(':')[0]) + (int(x.split(':')[1])/60))
df.info()

C:\Users\sidne\AppData\Local\Temp\ipykernel_5792\316595767.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())


<class 'pandas.core.frame.DataFrame'>
Index: 342434 entries, 0 to 141490
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   season_year              342434 non-null  object 
 1   game_date                342434 non-null  object 
 2   gameId                   342434 non-null  int64  
 3   matchup                  342434 non-null  object 
 4   teamId                   342434 non-null  int64  
 5   teamCity                 342434 non-null  object 
 6   teamName                 342434 non-null  object 
 7   teamTricode              342434 non-null  object 
 8   teamSlug                 342434 non-null  object 
 9   personId                 342434 non-null  int64  
 10  personName               342434 non-null  object 
 11  position                 342434 non-null  object 
 12  minutes                  342434 non-null  float64
 13  fieldGoalsMade           342434 non-null  int64  
 14  fieldGoal

In [171]:
player_stats = df.drop(columns = ['game_date', 'gameId', 'matchup', 'teamId', 'teamCity', 'teamTricode', 'teamSlug', 'personId'], axis = 1).copy()
player_stats = player_stats.groupby(by = ['personName', 'teamName', 'season_year', 'position']).mean(numeric_only=True).reset_index()
player_stats['fieldGoalsPercentage'] = player_stats.apply(lambda x: (x['fieldGoalsMade'] / x['fieldGoalsAttempted']) * 100 if x['fieldGoalsAttempted'] != 0 else 0, axis = 1)
player_stats['threePointersPercentage'] = player_stats.apply(lambda x: (x['threePointersMade'] / x['threePointersAttempted']) * 100 if x['threePointersAttempted'] != 0 else 0, axis = 1)
player_stats['freeThrowsPercentage'] = player_stats.apply(lambda x: (x['freeThrowsMade'] / x['freeThrowsAttempted']) * 100 if x['freeThrowsAttempted'] != 0 else 0, axis = 1)
player_stats.to_csv('../dataset/clean/stats_by_player.csv', index = False)

,personName,teamName,season_year,position,minutes,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,threePointersMade,threePointersAttempted,...,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints
6128,LeBron James,Cavaliers,2014-15,F,36.136715,9.043478,18.536232,48.788116,1.739130,4.913043,...,0.739130,5.289855,6.028986,7.405797,1.579710,0.710145,3.942029,1.956522,25.260870,7.840580
6129,LeBron James,Cavaliers,2015-16,F,35.638816,9.697368,18.631579,52.048023,1.144737,3.710526,...,1.460526,5.973684,7.434211,6.763158,1.368421,0.644737,3.276316,1.881579,25.263158,8.131579
6130,LeBron James,Cavaliers,2016-17,F,37.764640,9.945946,18.162162,54.761905,1.675676,4.621622,...,1.310811,7.324324,8.635135,8.729730,1.243243,0.594595,4.094595,1.810811,26.405405,6.527027
6131,LeBron James,Cavaliers,2017-18,F,36.838125,10.400000,19.250000,54.025974,1.800000,4.975000,...,1.200000,7.512500,8.712500,9.012500,1.412500,0.837500,4.212500,1.675000,27.350000,1.250000
6132,LeBron James,Cavaliers,2017-18,G,39.308333,12.500000,20.000000,62.500000,2.500000,4.000000,...,0.500000,5.500000,6.000000,13.000000,1.500000,2.000000,5.000000,1.000000,31.500000,2.500000
6133,LeBron James,Heat,2010-11,F,38.772363,9.594937,18.797468,51.043771,1.164557,3.531646,...,1.012658,6.455696,7.468354,7.012658,1.569620,0.632911,3.594937,2.063291,26.721519,7.835443
6134,LeBron James,Heat,2011-12,F,37.510656,9.934426,18.754098,52.972028,0.868852,2.409836,...,1.491803,6.475410,7.967213,6.278689,1.836066,0.803279,3.426230,1.573770,26.918033,7.573770
6135,LeBron James,Heat,2011-12,G,38.066667,15.000000,25.000000,60.000000,1.000000,2.000000,...,3.000000,3.000000,6.000000,4.000000,3.000000,1.000000,4.000000,0.000000,41.000000,12.000000
6136,LeBron James,Heat,2012-13,F,37.855921,10.065789,17.815789,56.499261,1.355263,3.342105,...,1.276316,6.750000,8.026316,7.250000,1.697368,0.881579,2.973684,1.447368,26.789474,9.473684
6137,LeBron James,Heat,2013-14,F,37.686580,9.961039,17.571429,56.688840,1.506494,3.974026,...,1.051948,5.870130,6.922078,6.337662,1.571429,0.337662,3.506494,1.636364,27.129870,5.311688
